# Train Q50, then Q10, from one shared local cache

This is the full registered comparison, not a smoke run. It first trains the paper-style **Q50 critic** for 8,000 updates and then automatically trains the causal **Q10 critic** for 8,000 updates. Both runs use the same immutable snapshot and the same locally mirrored compressed Supabase parts. Q10 therefore does not download the dataset again.

Training statistics print every 100 updates, validation statistics every 500 updates, and checkpoints are written to Drive every 1,000 updates. Rerunning this notebook in the same runtime reuses the local source cache; training resumes independently from the newest Q50/Q10 Drive checkpoints. No uncertainty signal enters either critic.

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pathlib import Path
import shutil
from google.colab import drive

drive.mount('/content/drive')

SNAPSHOT_ID = 'PASTE_PCPCDS_SNAPSHOT_ID'  # immutable snapshot from notebook 56
HORIZONS = (50, 10)                       # queued in this order
MICRO_BATCH_SIZE = {50: 8, 10: 16}       # effective batch is 64 via accumulation
CACHE_DOWNLOAD_WORKERS = 8               # independent Supabase clients
CACHE_ROOT = '/content/qplanning_cache'  # ephemeral local SSD, never Drive
OUTPUT_ROOT = '/content/drive/MyDrive/pnp_qplanning_corrector'

assert SNAPSHOT_ID.startswith('pcpcds-'), 'Paste the notebook-56 snapshot ID'
disk = shutil.disk_usage('/content')
print({
    'snapshot_id': SNAPSHOT_ID,
    'queued_horizons': HORIZONS,
    'run_mode': 'full',
    'updates_per_horizon': 8000,
    'train_print_every': 100,
    'validate_every': 500,
    'checkpoint_every': 1000,
    'download_workers': CACHE_DOWNLOAD_WORKERS,
    'local_free_GiB': round(disk.free / 2**30, 1),
    'cache_root': CACHE_ROOT,
    'output_root': OUTPUT_ROOT,
})

## Run both critics

The first call creates source_parts_v2 by copying the required compressed multipart objects directly to local disk. Its progress lines report completed rollouts, cached GiB, download rate, and ETA. Q50 windows are reconstructed in RAM by rollout. After Q50 finishes, GPU memory is cleared and Q10 starts from the same local parts without another remote download.

In [ ]:
import gc
import torch
from pnp.qplanning_critic import run_qplanning_training_test

reports = {}
for position, horizon in enumerate(HORIZONS, 1):
    print('\n' + '=' * 88)
    print(f'QUEUED RUN {position}/{len(HORIZONS)}: FULL Q{horizon}')
    print('=' * 88, flush=True)
    reports[horizon] = run_qplanning_training_test(
        snapshot_id=SNAPSHOT_ID,
        horizon=horizon,
        run_mode='full',
        cache_root=CACHE_ROOT,
        output_root=OUTPUT_ROOT,
        micro_batch_size=MICRO_BATCH_SIZE[horizon],
        cache_download_workers=CACHE_DOWNLOAD_WORKERS,
        stream_windows=True,
        resume=True,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'FULL Q{horizon} COMPLETE:', reports[horizon]['final_checkpoint'], flush=True)

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    'critic': f'Q{horizon}',
    'updates': report['updates'],
    'train_windows': report['train_windows'],
    'validation_windows': report['validation_windows'],
    **report['validation'],
    'checkpoint': report['final_checkpoint'],
} for horizon, report in reports.items()])
display(summary)

history = pd.concat([
    pd.DataFrame([{
        'critic': f'Q{horizon}', 'update': item['update'],
        **item['validation'],
    } for item in report['history']])
    for horizon, report in reports.items()
], ignore_index=True)
display(history)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for critic, rows in history.groupby('critic'):
    rows = rows.sort_values('update')
    axes[0].plot(rows.update, rows.failure_auc, marker='o', label=critic)
    axes[1].plot(rows.update, rows.return_mae, marker='o', label=critic)
axes[0].axhline(0.5, color='black', linestyle='--', linewidth=1)
axes[0].set(title='Validation failure AUC', xlabel='training update', ylabel='AUC')
axes[1].set(title='Validation return MAE', xlabel='training update', ylabel='MAE')
for axis in axes:
    axis.grid(alpha=.25)
    axis.legend()
plt.tight_layout()
plt.show()